# USER BEHAVIOR PREDICTION — DATAFLOW 2026
## Transformer model

### Import library and Initial Config

Khai báo các thư viện cần thiết trong quá trình huấn luyện mô hình, ngoài ra khai báo các biến, đường dẫn thư mục, siêu tham số cho mô hình

In [1]:
import os, random, math, argparse, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

warnings.filterwarnings("ignore")

In [2]:
CFG = {
    # ── Dữ liệu
    "x_train": "/kaggle/input/datasets/hungbaqv/user-behavior-prediction/X_train.csv",
    "y_train": "/kaggle/input/datasets/hungbaqv/user-behavior-prediction/Y_train.csv",
    "x_val": "/kaggle/input/datasets/hungbaqv/user-behavior-prediction/X_val.csv",
    "y_val": "/kaggle/input/datasets/hungbaqv/user-behavior-prediction/Y_val.csv",
    "x_test": "/kaggle/input/datasets/hungbaqv/user-behavior-prediction/X_test.csv",
    "output_file": "/kaggle/working/submission_transformer.csv",
    "save_path": "/kaggle/working/trans_model.pt",
    # ── Chuỗi
    "max_seq_len" : 37,  # Giữ nguyên
    "min_freq" : 2,  # Giữ nguyên
    # ── Transformer architecture
    "model_type" : "transformer",
    "embed_dim" : 512,  # Tăng từ 256 để capacity cao hơn
    "hidden_dim" : 512,  # Tăng để FFN dim lớn hơn
    "n_layers" :8,  # Tăng từ 4 để depth tốt hơn
    "n_heads" : 8,  # Giữ nguyên
    "dropout" : 0.1,  # Giảm từ 0.2 để retain info
    "head_hidden" : 256,  # Giữ nguyên
    "focal_gamma": 1.0,  # Giữ, nhưng test 3.0 nếu imbalance cao
    "label_smoothing": 0.1,  # Thêm để tune trong loss
    # ── Training — Transformer cần khác BiLSTM
    "batch_size" : 64,  # Giảm từ 128 cho imbalance
    "epochs" : 100,  # Giảm từ 100 để tránh overfit
    "lr" : 5e-5,  # Giảm từ 5e-4 cho stability
    "weight_decay": 0.02,  # Giữ, nhưng test 0.01 nếu underfit
    "warmup_epochs": 5,  # Giảm từ 15, tương đương pct_start=0.1
    "patience" : 10,  # Giảm từ 15 cho early stop nhanh
    "seed" : 42,
    "num_workers" : 0,
    "device" : "cuda" if torch.cuda.is_available() else "cpu",
    # Thêm cho tuning
    "use_optuna": False,  # Set True để enable Optuna
}
FEATURE_COLS = [f"feature_{i}" for i in range(1, 38)]
TARGET_COLS = [f"attr_{i}" for i in range(1, 7)]

In [ ]:
def seed_everything(seed: int):
    """
    Đặt seed cho tất cả các thư viện để đảm bảo kết quả có thể tái lập.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG["seed"])

### Xây dựng Vocab và Chuẩn bị dữ liệu Dataloader cho mô hình

In [4]:
class Vocabulary:
    """
    Map token gốc → index liên tục.
    Index 0 = PAD, Index 1 = UNK.
    min_freq: token xuất hiện ít hơn ngưỡng này → UNK
              giúp model học cách xử lý token lạ từ đầu.
    """
    PAD_IDX = 0
    UNK_IDX = 1

    def __init__(self):
        self.token2idx = {}
        self.next_idx  = 2

    def build(self, X: pd.DataFrame, min_freq: int = 1):
        from collections import Counter
        freq = Counter()
        for col in FEATURE_COLS:
            freq.update(X[col].dropna().values)

        for token, count in freq.items():
            if count >= min_freq:
                self.token2idx[token] = self.next_idx
                self.next_idx += 1

        n_unk = sum(1 for c in freq.values() if c < min_freq)
        print(f"  Vocab size : {self.next_idx} tokens")
        print(f"  → Kept     : {self.next_idx - 2} | UNK'd: {n_unk} (min_freq={min_freq})")

    def encode(self, val) -> int:
        return self.token2idx.get(val, self.UNK_IDX)

    @property
    def size(self) -> int:
        return self.next_idx

In [5]:
class BehaviorDataset(Dataset):
    """
    RIGHT-PAD: token thực nằm bên TRÁI, PAD bên PHẢI.
    Transformer xử lý tốt cả hai nhờ src_key_padding_mask.
    """
    def __init__(self,
                 X: pd.DataFrame,
                 vocab: Vocabulary,
                 max_seq_len: int,
                 Y: pd.DataFrame = None,
                 n_classes: list  = None):
        self.X, self.Y, self.vocab = X, Y, vocab
        self.max_seq_len = max_seq_len
        self.ids = X.index.tolist()

        self.label_encoders = {}
        if Y is not None and n_classes is not None:
            for col in TARGET_COLS:
                uniq = sorted(Y[col].unique())
                self.label_encoders[col] = {v: j for j, v in enumerate(uniq)}

    def __len__(self):
        return len(self.ids)

    def _encode_row(self, idx):
        tokens_raw = self.X.loc[idx, FEATURE_COLS].dropna().values
        seq_len    = len(tokens_raw)
        encoded    = [self.vocab.encode(t) for t in tokens_raw]

        # Truncate: giữ phần CUỐI (recent context quan trọng hơn)
        if seq_len > self.max_seq_len:
            encoded = encoded[-self.max_seq_len:]
            seq_len = self.max_seq_len

        # ✅ RIGHT-PAD: token thực bên trái, PAD bên phải
        pad_len = self.max_seq_len - seq_len
        seq     = encoded + [self.vocab.PAD_IDX] * pad_len
        mask    = [True] * seq_len + [False] * pad_len

        # pad_len = self.max_seq_len - seq_len
        # seq  = [self.vocab.PAD_IDX] * pad_len + encoded
        # mask = [False] * pad_len + [True] * seq_len

        # Meta features
        unique_ratio = len(set(encoded)) / max(seq_len, 1)
        # first_tok    = (encoded[0] / self.vocab.size) if seq_len > 0 else 0.0
        seq_len_norm = seq_len / self.max_seq_len

        return seq, mask, [seq_len_norm, unique_ratio]

    def __getitem__(self, i):
        idx = self.ids[i]
        seq, mask, meta = self._encode_row(idx)

        seq_t  = torch.tensor(seq,  dtype=torch.long)
        mask_t = torch.tensor(mask, dtype=torch.bool)
        meta_t = torch.tensor(meta, dtype=torch.float)

        if self.Y is not None:
            labels = [
                self.label_encoders[col].get(self.Y.loc[idx, col], 0)
                for col in TARGET_COLS
            ]
            return seq_t, mask_t, meta_t, torch.tensor(labels, dtype=torch.long)

        return seq_t, mask_t, meta_t


### Khai báo các Class liên quan đến kiến trúc của mô hình

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ClassificationHead(nn.Module):
    """
    MLP 2 lớp cho mỗi attr: [context_dim + meta_dim] → n_classes
    Sử dụng sau khi đã gộp context vector và meta-features.
    """
    def __init__(self, in_dim, hidden_dim, n_classes, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, x):
        return self.net(x)

class TransformerEncoder(nn.Module):
    """
    Mã hoá chuỗi bằng Transformer Encoder, trả về vector CLS.
    Thêm positional embedding và segment embedding để model hiểu được vị trí và phân đoạn của token.
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim,
                 n_layers, n_heads, max_seq_len, dropout):
        super().__init__()
        self.max_seq_len = max_seq_len

        self.embedding     = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_seq_len + 1, embed_dim) 
        self.seg_embedding = nn.Embedding(5, embed_dim)
        self.dropout        = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = embed_dim,
            nhead           = n_heads,
            dim_feedforward = embed_dim * 4, # Thường dùng x4 để Transformer mạnh hơn
            dropout         = dropout,
            batch_first     = True,
            activation      = F.gelu, # Sử dụng hàm GELU trực tiếp
            norm_first      = True,
        )
        self.encoder   = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.out_dim   = embed_dim

    def _segment_ids(self, mask: torch.BoolTensor) -> torch.LongTensor:
        # Giữ nguyên logic vectorized cực tốt của bạn
        cumsum = mask.cumsum(dim=1).float()
        total  = mask.sum(dim=1, keepdim=True).float()
        ratio  = cumsum / (total + 1e-9)
        seg = torch.full(mask.shape, 4, dtype=torch.long, device=mask.device)
        seg[mask & (ratio <= 0.33)] = 1
        seg[mask & (ratio > 0.33) & (ratio <= 0.67)] = 2
        seg[mask & (ratio > 0.67)] = 3
        return seg

    def forward(self, seq, mask):
        B, T = seq.shape
        # Embeddings
        x = self.embedding(seq)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)

        # Positional & Segment
        pos = torch.arange(T + 1, device=seq.device).unsqueeze(0).expand(B, -1)
        pos_emb = self.pos_embedding(pos)

        cls_seg = torch.zeros(B, 1, dtype=torch.long, device=seq.device)
        tok_seg = self._segment_ids(mask)
        seg_emb = self.seg_embedding(torch.cat([cls_seg, tok_seg], dim=1))

        x = self.dropout(x + pos_emb + seg_emb)

        # Masking
        cls_keep = torch.zeros(B, 1, dtype=torch.bool, device=mask.device)
        src_key_padding_mask = torch.cat([cls_keep, ~mask], dim=1)

        out = self.encoder(x, src_key_padding_mask=src_key_padding_mask)
        return out[:, 0, :] # Trả về CLS vector

class UserBehaviorModel(nn.Module):
    """
    Cải tiến chính: Thêm lớp 'Neck' (Projection) để kết hợp Meta features 
    một cách hiệu quả hơn trước khi chia vào 6 đầu phân loại.
    """
    def __init__(self, vocab_size: int, n_classes_list: list, cfg: dict):
        super().__init__()
        self.encoder = TransformerEncoder(
            vocab_size  = vocab_size,
            embed_dim   = cfg["embed_dim"],
            hidden_dim  = cfg["hidden_dim"],
            n_layers    = cfg["n_layers"],
            n_heads     = cfg["n_heads"],
            max_seq_len = cfg["max_seq_len"],
            dropout     = cfg["dropout"],
        )

        # Feature Fusion Neck
        combined_dim = cfg["embed_dim"] + 2
        self.neck = nn.Sequential(
            nn.Linear(combined_dim, cfg["embed_dim"]),
            nn.LayerNorm(cfg["embed_dim"]),
            nn.GELU(),
            nn.Dropout(cfg["dropout"])
        )

        self.heads = nn.ModuleList([
            ClassificationHead(cfg["embed_dim"], cfg["head_hidden"], n_cls, cfg["dropout"])
            for n_cls in n_classes_list
        ])

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02) # Truncated normal thường tốt hơn cho Transformer
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, seq, mask, meta):
        # 1. Trích xuất đặc trưng chuỗi
        context = self.encoder(seq, mask) # [B, embed_dim]
        
        # 2. Kết hợp với meta features
        combined = torch.cat([context, meta], dim=-1) # [B, embed_dim + 3]
        
        # 3. Đi qua Neck để nén thông tin
        fused_features = self.neck(combined) # [B, embed_dim]
        
        # 4. Phân loại đa mục tiêu
        return [head(fused_features) for head in self.heads]

### Xây dựng hàm loss để tối ưu hóa mô hình

Trong quá trình huấn luyện mô hình nhóm có sử dụng các hàm loss khác nhau vì vậy cần khai báo nhiều class loss

In [7]:
class UncertaintyWeightedLoss(nn.Module):
    """
    Uncertainty Weighting cho multi-output classification (Kendall et al., 2018).
    - Mỗi task có một tham số log_var (σ²) học được.
    - Loss của task i: (1/(2σ_i²)) * L_i + log(σ_i)
    - Tổng loss = sum over tasks + regularization term.
    - Áp dụng cho classification: dùng CrossEntropyLoss cho từng head.
    """
    def __init__(self, num_tasks: int = 6, device: str = "cuda"):
        super().__init__()
        # log_var: tham số uncertainty (σ²), khởi tạo = 0 → σ=1 ban đầu
        self.log_vars = nn.Parameter(torch.zeros(num_tasks, device=device), requires_grad=True)
        self.loss_fns = nn.ModuleList([
            nn.CrossEntropyLoss(label_smoothing=CFG.get("label_smoothing", 0.1))
            for _ in range(num_tasks)
        ])
        self.num_tasks = num_tasks
        self.device = device

    def forward(self, logits_list, targets):
        """
        logits_list: list of 6 tensors [B, n_classes_i]
        targets: [B, 6]
        Returns: total_loss, individual_losses (để log)
        """
        total_loss = 0.0
        losses = []

        for i in range(self.num_tasks):
            # Loss gốc của task i (CrossEntropy)
            task_loss = self.loss_fns[i](logits_list[i], targets[:, i])

            # Uncertainty weighting (equation 7 in paper, for classification)
            # precision = 1 / (2 * σ_i²) = 1 / (2 * exp(log_var_i))
            precision = 0.5 * torch.exp(-self.log_vars[i])  # 1/(2σ²)
            uncertainty_term = 0.5 * self.log_vars[i]       # log(σ) = 0.5 * log_var (vì log(σ²)=log_var → log(σ)=0.5*log_var)

            weighted_loss = precision * task_loss + uncertainty_term

            total_loss += weighted_loss
            losses.append(task_loss.item())  # log loss gốc để theo dõi

        # Optional: regularization để tránh log_var quá nhỏ (σ→0)
        # total_loss += 0.001 * self.log_vars.abs().sum()  # nếu cần

        return total_loss, losses

    def get_uncertainties(self):
        """Trả về σ_i = exp(0.5 * log_var_i) cho từng task"""
        return torch.exp(0.5 * self.log_vars).detach().cpu().numpy().tolist()

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiOutputLoss(nn.Module):
    """
    Loss lý tưởng: Multi-Task + Focal + Label Smoothing.
    - Focal cho từng head để xử lý imbalance.
    - Weights giữa heads (ưu tiên head khó).
    - Class weights trong Focal cho imbalance class.
    """
    def __init__(self, n_classes_list, weights=None, label_smoothing=0.1, 
                 class_weights_list=None, gamma=2.0):
        super().__init__()
        if weights is None:
            self.weights = [1.0] * len(n_classes_list)
        else:
            self.weights = weights
        
        # List FocalLoss cho từng head, với alpha=class_weights và gamma
        self.loss_fns = nn.ModuleList([
            FocalLoss(
                gamma=gamma,
                alpha=class_weights_list[i] if class_weights_list else None,
                label_smoothing=label_smoothing,
                reduction='mean'  # Mean để ổn định
            ) for i in range(len(n_classes_list))
        ])
        
        print(f"🏗️ MultiOutputLoss (Focal + Smoothing): Gamma={gamma}, Smoothing={label_smoothing}")
        for i, w in enumerate(self.weights):
            print(f"  - Head {i+1} (attr_{i+1}) weight: {w}")
            if class_weights_list:
                print(f"    Class weights: {class_weights_list[i].tolist()}")

    def forward(self, logits_list, targets):
        individual_losses = []
        weighted_losses = []

        for i, lg in enumerate(logits_list):
            target = targets[:, i]
            loss = self.loss_fns[i](lg, target)  # FocalLoss riêng cho head i
            individual_losses.append(loss.item())
            weighted_losses.append(loss * self.weights[i])

        total_loss = sum(weighted_losses) / len(weighted_losses)
        return total_loss, individual_losses

In [9]:
class FocalLoss(nn.Module):
    """
    Focal Loss với hỗ trợ class weights (alpha), label smoothing, và reduction.
    Formula: FL(pt) = -alpha * (1 - pt)^gamma * log(pt)
    Kết hợp với CE có smoothing.
    """
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha  # Tensor class weights (nếu có)
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Tính CE với smoothing và weights (nếu alpha là tensor)
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', 
                                 label_smoothing=self.label_smoothing, weight=self.alpha)
        
        # Tính pt = exp(-ce_loss)
        pt = torch.exp(-ce_loss)
        
        # Focal factor: (1 - pt)^gamma
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        
        # Nếu reduction là mean
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

### Một số hàm đơn hỗ trợ vào các quá trình training model

In [10]:
def get_warmup_cosine_scheduler(optimizer, warmup_epochs: int, total_epochs: int):
    """
    - Epoch 0 .. warmup_epochs-1 : lr tăng tuyến tính từ 0 → lr
    - Epoch warmup_epochs .. end  : lr giảm theo Cosine về ~0
    Transformer cần warmup để attention weights ổn định trước
    khi nhận gradient lớn.
    """
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            # Tăng dần: tránh gradient shock ở các epoch đầu
            return (epoch + 1) / warmup_epochs
        # Cosine decay sau warmup
        progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return LambdaLR(optimizer, lr_lambda)

In [11]:
def exact_match_accuracy(preds: np.ndarray, labels: np.ndarray) -> float:
    return (preds == labels).all(axis=1).mean()

def per_attr_accuracy(preds: np.ndarray, labels: np.ndarray) -> dict:
    return {f"attr_{i+1}_acc": (preds[:, i] == labels[:, i]).mean()
            for i in range(6)}

In [ ]:
def load_csv(path):
    return pd.read_csv(path, index_col="id")

def compute_class_info(Y: pd.DataFrame):
    n_classes_list, imbalance_ratios, class_maps = [], [], {}
    class_weights_list = []  # List of tensors for each attr
    for col in TARGET_COLS:
        vc = Y[col].value_counts()
        uniq = sorted(Y[col].unique())
        n_classes_list.append(len(uniq))
        ir = vc.max() / vc.min() if vc.min() > 0 else float("inf")
        imbalance_ratios.append(ir)
        class_maps[col] = {j: v for j, v in enumerate(uniq)}
        
        # Tính class weights: 1 / freq, normalized
        freq = np.array([vc.get(u, 1) for u in uniq])  # Freq cho từng class, min=1 để tránh inf
        weights = 1.0 / freq
        weights = weights / weights.sum() * len(uniq)  # Normalize để mean ~1
        class_weights_list.append(torch.tensor(weights, dtype=torch.float))
    
    return n_classes_list, imbalance_ratios, class_maps, class_weights_list

### Các hàm chạy toàn bộ pipeline training

In [12]:
def train_one_epoch(model, loader, optimizer, loss_fn, device, epoch=None):
    """
    Train một epoch. Train EM được tính SAU khi update xong
    bằng cách chạy lại ở eval mode → số liệu chính xác hơn.
    """
    model.train()
    total_loss = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Train]", leave=False)
    for batch in pbar:
        seq, mask, meta, labels = [b.to(device) for b in batch]
        optimizer.zero_grad()
        logits = model(seq, mask, meta)
        loss, _ = loss_fn(logits, labels)
        loss.backward()
        # Gradient clipping — quan trọng với Transformer
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    # ✅ Tính Train EM ở eval mode → không bị ảnh hưởng bởi Dropout
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            seq, mask, meta, labels = [b.to(device) for b in batch]
            logits = model(seq, mask, meta)
            preds  = torch.stack([l.argmax(-1) for l in logits], dim=-1)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    model.train()  # Trả về train mode

    preds_np  = np.concatenate(all_preds)
    labels_np = np.concatenate(all_labels)
    return total_loss / len(loader), exact_match_accuracy(preds_np, labels_np)


@torch.no_grad()
def evaluate(model, loader, loss_fn, device, epoch=None):
    model.eval()
    total_loss  = 0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Val]", leave=False)

    for batch in pbar:
        seq, mask, meta, labels = [b.to(device) for b in batch]
        logits = model(seq, mask, meta)
        loss, _ = loss_fn(logits, labels)
        total_loss += loss.item()
        preds = torch.stack([l.argmax(-1) for l in logits], dim=-1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    preds_np  = np.concatenate(all_preds)
    labels_np = np.concatenate(all_labels)
    return (total_loss / len(loader),
            exact_match_accuracy(preds_np, labels_np),
            per_attr_accuracy(preds_np, labels_np),
            preds_np)


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    all_preds = []
    for batch in loader:
        seq, mask, meta = [b.to(device) for b in batch]
        logits = model(seq, mask, meta)
        preds  = torch.stack([l.argmax(-1) for l in logits], dim=-1)
        all_preds.append(preds.cpu().numpy())
    return np.concatenate(all_preds)


In [14]:
def train_one_epoch_v2(model, loader, optimizer, loss_fn, device, scheduler=None):
    model.train()
    total_loss = 0
    total_em = 0
    
    pbar = tqdm(loader, desc="Training", leave=False)
    
    for seq, mask, meta, labels in pbar:
        seq, mask, meta, labels = seq.to(device), mask.to(device), meta.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        logits_list = model(seq, mask, meta)
        loss, _ = loss_fn(logits_list, labels)
        
        # Backward pass
        loss.backward()
        
        # ✅ Gradient Clipping: Cực kỳ quan trọng cho Transformer để tránh nổ gradient
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # ✅ Step scheduler sau mỗi batch (Yêu cầu bắt buộc của OneCycleLR)
        if scheduler is not None:
            scheduler.step()
            
        # Tính toán Exact Match (EM)
        batch_size = seq.size(0)
        preds = torch.stack([torch.argmax(lg, dim=-1) for lg in logits_list], dim=1)
        correct = (preds == labels).all(dim=1).sum().item()
        
        total_loss += loss.item() * batch_size
        total_em += correct
        
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{optimizer.param_groups[0]['lr']:.2e}"})
        
    return total_loss / len(loader.dataset), total_em / len(loader.dataset)

In [15]:
def run_training(cfg, X_train, Y_train, X_val=None, Y_val=None):
    device = cfg["device"]
    print(f"\n🖥️  Device  : {device}")
    print(f"🔧  Model   : TRANSFORMER (Hyper-optimized)")
    
    # ── 1. Chuẩn bị Dữ liệu
    n_classes_list, imbalance_ratios, class_maps, class_weights_list = compute_class_info(Y_train)
    class_weights_list = [w.to(device) for w in class_weights_list]
    vocab = Vocabulary()
    vocab.build(X_train, min_freq=cfg.get("min_freq", 1))

    train_ds = BehaviorDataset(X_train, vocab, cfg["max_seq_len"], Y_train, n_classes_list)
    train_dl = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, 
                          num_workers=cfg["num_workers"], pin_memory=True)

    val_dl = None
    if X_val is not None and Y_val is not None:
        val_ds = BehaviorDataset(X_val, vocab, cfg["max_seq_len"], Y_val, n_classes_list)
        val_dl = DataLoader(val_ds, batch_size=cfg["batch_size"] * 2, shuffle=False)

    # ── 2. Khởi tạo Model & Optimizer
    model = UserBehaviorModel(vocab.size, n_classes_list, cfg).to(device)
    
    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         "weight_decay": cfg["weight_decay"]},
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
         "weight_decay": 0.0},
    ]
    optimizer = AdamW(optimizer_grouped_parameters, lr=cfg["lr"], betas=(0.9, 0.98), eps=1e-8)

    # ── 3. Scheduler
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg["lr"], steps_per_epoch=len(train_dl), epochs=cfg["epochs"],
        pct_start=0.15, anneal_strategy='cos'
    )

    # ── 4. Loss lý tưởng: Multi-Task + Focal + Smoothing
    # Head weights: Ưu tiên head imbalance cao
    total_ir = sum(imbalance_ratios)
    head_weights = [ir / total_ir if total_ir > 0 else 1.0 / len(imbalance_ratios) for ir in imbalance_ratios]
    
    loss_fn = MultiOutputLoss(n_classes_list, weights=head_weights, label_smoothing=0.1, 
                              class_weights_list=class_weights_list, gamma=cfg.get("focal_gamma", 2.0))
    # loss_fn = UncertaintyWeightedLoss(num_tasks=6, device=device)

    # ── 5. Training Loop (giữ nguyên phần còn lại)
    best_val_em = -1
    best_train_em = -1
    patience_cnt = 0
    history = defaultdict(list)

    print(f"\n🚀 Khởi chạy Training với {cfg['epochs']} Epochs...")
    print(f"{'Epoch':>5} | {'Tr. Loss':>9} | {'Tr. EM':>8} | {'Val Loss':>8} | {'Val EM':>7} | {'LR':>9}")
    print("-" * 72)

    for epoch in range(1, cfg["epochs"] + 1):
        train_loss, train_em = train_one_epoch_v2(model, train_dl, optimizer, loss_fn, device, scheduler)
        
        history["train_loss"].append(train_loss)
        history["train_em"].append(train_em)
        current_lr = optimizer.param_groups[0]['lr']

        if val_dl is not None:
            val_loss, val_em, per_attr, _ = evaluate(model, val_dl, loss_fn, device)
            history["val_loss"].append(val_loss)
            history["val_em"].append(val_em)

            status = "⭐ NEW BEST" if val_em > best_val_em else ""
            print(f"{epoch:>5} | {train_loss:>9.4f} | {train_em:>8.4f} | {val_loss:>8.4f} | {val_em:>7.4f} | {current_lr:.2e} {status}")

            if val_em > best_val_em:
                best_val_em = val_em
                patience_cnt = 0
                torch.save({"model_state": model.state_dict(), "cfg": cfg, "vocab": vocab}, cfg["save_path"])
            else:
                patience_cnt += 1
                if patience_cnt >= cfg["patience"]:
                    print(f"\n⏹️ Early stopping tại epoch {epoch}")
                    break
        else:
            status = ""

            if train_em > best_train_em:
                best_train_em = train_em
                status = "⭐ NEW BEST"
                
                torch.save(
                    {
                        "model_state": model.state_dict(),
                        "cfg": cfg,
                        "vocab": vocab
                    },
                    cfg["save_path"]
                )
        
            print(
                f"{epoch:>5} | {train_loss:>9.4f} | {train_em:>8.4f} |   N/A    |   N/A   | {current_lr:.2e} {status}"
            )

    return model, vocab, n_classes_list, class_maps, history

In [16]:
def make_submission(cfg, X_test, model, vocab, n_classes_list, class_maps):
    device = cfg["device"]
    model.to(device).eval()

    test_ds = BehaviorDataset(X_test, vocab, cfg["max_seq_len"])
    test_dl = DataLoader(test_ds, batch_size=cfg["batch_size"] * 2,
                         shuffle=False, num_workers=cfg["num_workers"])

    preds_idx = predict(model, test_dl, device)   # [N, 6]

    preds_raw = np.zeros_like(preds_idx, dtype=np.uint16)
    for i, col in enumerate(TARGET_COLS):
        preds_raw[:, i] = [class_maps[col].get(idx, 0) for idx in preds_idx[:, i]]

    sub = pd.DataFrame(preds_raw, columns=TARGET_COLS, index=X_test.index)
    sub.index.name = "id"
    sub = sub.astype(np.uint16)
    sub.to_csv(cfg["output_file"])
    print(f"\n✅ Submission saved: {cfg['output_file']}")
    print(sub.head())
    return sub

In [17]:
def main(args):
    # parser = argparse.ArgumentParser()
    # parser.add_argument("--mode", choices=["train", "submit"], default="train")
    # parser.add_argument("--epochs",      type=int,   default=CFG["epochs"])
    # parser.add_argument("--batch_size",  type=int,   default=CFG["batch_size"])
    # parser.add_argument("--lr",          type=float, default=CFG["lr"])
    # parser.add_argument("--warmup",      type=int,   default=CFG["warmup_epochs"])
    # parser.add_argument("--max_seq_len", type=int,   default=CFG["max_seq_len"])
    # parser.add_argument("--patience",    type=int,   default=CFG["patience"])
    # args = parser.parse_args()

    CFG.update({
        "epochs"       : args.epochs,
        "batch_size"   : args.batch_size,
        "lr"           : args.lr,
        "warmup_epochs": args.warmup,
        "max_seq_len"  : args.max_seq_len,
        "patience"     : args.patience,
    })

    print("=" * 68)
    print("  USER BEHAVIOR PREDICTION — TRANSFORMER (OPTIMIZED)")
    print(f"  Mode: {args.mode.upper()}")
    print("=" * 68)

    X_train = load_csv(CFG["x_train"]); Y_train = load_csv(CFG["y_train"])
    X_val   = load_csv(CFG["x_val"]);   Y_val   = load_csv(CFG["y_val"])

    if args.mode == "train":
        print(f"\n[Giai đoạn 1] Train={len(X_train)} | Val={len(X_val)}")
        model, vocab, n_classes_list, class_maps, _ = run_training(
            CFG, X_train, Y_train, X_val, Y_val
        )

    elif args.mode == "submit":
        print("\n[Giai đoạn 2] Gộp Train+Val...")
        X_all = pd.concat([X_train, X_val])
        Y_all = pd.concat([Y_train, Y_val])
        print(f"  Tổng: {len(X_all)} mẫu")

        model, vocab, n_classes_list, class_maps, _ = run_training(
            CFG, X_all, Y_all
        )
        X_test = load_csv(CFG["x_test"])
        make_submission(CFG, X_test, model, vocab, n_classes_list, class_maps)

    print("\n✅ HOÀN THÀNH!")


In [18]:
from types import SimpleNamespace

# args = SimpleNamespace(
#     mode="train",
#     epochs=CFG["epochs"],
#     batch_size=CFG["batch_size"],
#     lr=CFG["lr"],
#     max_seq_len=CFG["max_seq_len"],
#     warmup=CFG["warmup_epochs"],
#     patience = CFG["patience"],
# )

# main(args)

In [ ]:
args = SimpleNamespace(
    mode="submit",
    epochs=CFG["epochs"],
    batch_size=CFG["batch_size"],
    lr=CFG["lr"],
    max_seq_len=CFG["max_seq_len"],
    warmup=CFG["warmup_epochs"],
    patience = CFG["patience"],
)

main(args)

  USER BEHAVIOR PREDICTION — TRANSFORMER (OPTIMIZED)
  Mode: SUBMIT

[Giai đoạn 2] Gộp Train+Val...
  Tổng: 58200 mẫu

🖥️  Device  : cuda
🔧  Model   : TRANSFORMER (Hyper-optimized)
  Vocab size : 256 tokens
  → Kept     : 254 | UNK'd: 0 (min_freq=2)
🏗️ MultiOutputLoss (Focal + Smoothing): Gamma=1.0, Smoothing=0.1
  - Head 1 (attr_1) weight: 0.4374557876831617
    Class weights: [0.14443452656269073, 2.928534507751465, 0.029531782492995262, 2.12318754196167, 0.05796307697892189, 2.4978675842285156, 1.8224785327911377, 0.31477946043014526, 0.10586824268102646, 1.9127815961837769, 0.03298155218362808, 0.02959146350622177]
  - Head 2 (attr_2) weight: 0.05670901417831797
    Class weights: [0.08351757377386093, 1.0736346244812012, 1.0497583150863647, 1.0435696840286255, 1.0497583150863647, 1.0435696840286255, 1.013689398765564, 1.0043432712554932, 1.0284124612808228, 1.0336672067642212, 1.067162036895752, 1.0528802871704102, 1.0232110023498535, 1.0129642486572266, 1.0420339107513428, 1.0552

Training:   0%|          | 0/910 [00:00<?, ?it/s]

    1 |    0.0823 |   0.0000 |   N/A    |   N/A   | 2.52e-06 ⭐ NEW BEST


Training:   0%|          | 0/910 [00:00<?, ?it/s]

    2 |    0.0810 |   0.0000 |   N/A    |   N/A   | 4.08e-06 


...

Training:   0%|          | 0/910 [00:00<?, ?it/s]

   19 |    0.0158 |   0.7393 |   N/A    |   N/A   | 4.97e-05 ⭐ NEW BEST


Training:   0%|          | 0/910 [00:00<?, ?it/s]

   20 |    0.0157 |   0.7663 |   N/A    |   N/A   | 4.96e-05 ⭐ NEW BEST


Training:   0%|          | 0/910 [00:00<?, ?it/s]

In [ ]:
# Y_train = pd.read_csv(CFG["y_train"])
# TARGET_COLS = [f"attr_{i}" for i in range(1, 7)]

# n_classes_list = []
# class_maps     = []   # list[dict]: index (int) → giá trị UINT16 thực

# for col in TARGET_COLS:
#     unique_vals = sorted(Y_train[col].unique())
#     n_classes_list.append(len(unique_vals))
#     class_maps.append({idx: int(val) for idx, val in enumerate(unique_vals)})

# print("n_classes_list:", n_classes_list)
# for i, cm in enumerate(class_maps):
#     sample = dict(list(cm.items())[:3])
#     print(f"  attr_{i+1}: {len(cm)} classes | sample map: {sample} ...")

# # ── 2. Load checkpoint cũ (nếu đã train) ────────────────────────
# ckpt_old = torch.load(CFG["save_path"], map_location="cpu", weights_only=False)

# # ── 3. Lưu checkpoint mới với đầy đủ thông tin ──────────────────
# NEW_SAVE_PATH = "/kaggle/working/trans_model_final.pt"

# torch.save(
#     {
#         "model_state":    ckpt_old["model_state"],
#         "cfg":            ckpt_old["cfg"],
#         "vocab":          ckpt_old["vocab"],
#         "n_classes_list": n_classes_list,   # ← THÊM MỚI
#         "class_maps":     class_maps,       # ← THÊM MỚI
#     },
#     NEW_SAVE_PATH,
# )
# print(f"\n✅ Đã lưu checkpoint mới tại: {NEW_SAVE_PATH}")
# print(f"   Tải file này về và đặt vào: backend/model_weights/model.pt")